In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
#@title installing I2MC and e
%pip install I2MC
%cd /content/drive/MyDrive/CAMES/pilot_testing/pilot_data/julie_1712/et_package
%pip install -e .

/content/drive/MyDrive/CAMES/pilot_testing/pilot_data/julie_1712/et_package
Obtaining file:///content/drive/MyDrive/CAMES/pilot_testing/pilot_data/julie_1712/et_package
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 19.0 MB/s eta 0:00:00
  Building editable for et_package (pyproject.toml) ... done
  Created wheel for et_package: filename=et_package-0.1-0.editable-py3-none-any.whl size=2800 sha256=6df0352776f40e84a906aa8ca9525022eac69f53c344cb75ec076eb538b3f78

In [1]:
import os
import ast
import math
import logging
import traceback

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import I2MC
from et_package.et_func import ET_Func

In [2]:
PARTICIPANTS =  ["P12"]#[f"P{i:02d}" for i in range(1, 21) if i not in (1, 2, 3, 4, 5, 6, 7, 11, 12, 13)]

BASE_DIR = "/content/drive/MyDrive/CAMES/data_collection_training"

SAVE_FIXATIONS = True
SAVE_SACCADES  = True

#Screen / hardware
SCREEN_SIZE_IN      = 24
SCREEN_RATIO        = 16 / 9
VIEWING_DISTANCE_CM = 60
SCREEN_RESOLUTION_X = 1920
SCREEN_RESOLUTION_Y = 1080
FREQ                = 90

#Logging
import importlib
importlib.reload(logging)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger(__name__)

#Derived constants
SCREEN_WIDTH_MM = SCREEN_SIZE_IN * 25.4 * (
    SCREEN_RESOLUTION_X / math.sqrt(SCREEN_RESOLUTION_X**2 + SCREEN_RESOLUTION_Y**2)
)
VIEWING_DISTANCE_MM = VIEWING_DISTANCE_CM * 10
px2deg = (
    math.atan2(0.5 * SCREEN_WIDTH_MM, VIEWING_DISTANCE_MM)
    / math.pi * 180
    / (0.5 * SCREEN_RESOLUTION_X)
)

et = ET_Func(
    screen_size       = SCREEN_SIZE_IN,
    screen_ratio      = SCREEN_RATIO,
    viewing_distance  = VIEWING_DISTANCE_CM,
    screen_resolution = SCREEN_RESOLUTION_X,
    freq              = FREQ,
)

In [3]:
# ── Helper functions ───────────────────────────────────────────────────────────
def split_xy_series(s: pd.Series):
    """Parse '(x, y)' string columns into separate float columns."""
    def parse(v):
        if pd.isna(v):
            return (np.nan, np.nan)
        if isinstance(v, (list, tuple)) and len(v) >= 2:
            return (float(v[0]), float(v[1]))
        if isinstance(v, str):
            try:
                vv = ast.literal_eval(v)
                if isinstance(vv, (list, tuple)) and len(vv) >= 2:
                    return (float(vv[0]), float(vv[1]))
            except Exception:
                pass
        return (np.nan, np.nan)
    parsed = s.apply(parse)
    return parsed.apply(lambda t: t[0]), parsed.apply(lambda t: t[1])


def load_and_clean(gaze_path: str) -> pd.DataFrame:
    """Load gaze_marked.csv, parse timestamps and XY coordinates."""
    gaze = pd.read_csv(gaze_path, low_memory=False)

    gaze["t_unix"]                 = pd.to_numeric(gaze["Timestamp Unix"], errors="raise")
    gaze["device_time_stamp_orig"] = pd.to_numeric(gaze["device_time_stamp"], errors="raise")
    gaze = gaze.sort_values("t_unix").reset_index(drop=True)
    gaze["time_s"] = (
        gaze["device_time_stamp_orig"] - gaze["device_time_stamp_orig"].iloc[0]
    ) / 1e6

    for side in ("left", "right"):
        col = f"{side}_gaze_point_on_display_area"
        gaze[f"{col}_0"], gaze[f"{col}_1"] = split_xy_series(gaze[col])

    for side in ("left", "right"):
        vcol = f"{side}_gaze_point_validity"
        xcol = f"{side}_gaze_point_on_display_area_0"
        ycol = f"{side}_gaze_point_on_display_area_1"
        if vcol in gaze.columns:
            gaze.loc[gaze[vcol] < 0.5, [xcol, ycol]] = np.nan

    return gaze


def compute_validity_rate(gaze: pd.DataFrame) -> float:
    """Fraction of samples where at least one eye has valid gaze."""
    left_valid  = gaze["left_gaze_point_validity"]  >= 0.5 if "left_gaze_point_validity"  in gaze.columns else pd.Series(False, index=gaze.index)
    right_valid = gaze["right_gaze_point_validity"] >= 0.5 if "right_gaze_point_validity" in gaze.columns else pd.Series(False, index=gaze.index)
    return float((left_valid | right_valid).mean())


def run_pipeline(gaze: pd.DataFrame):
    """Run fixation and saccade detection."""
    events, proc_gaze, _ = et.find_fix(gaze)
    fixations = pd.DataFrame(events)

    proc_gaze = et.calc_velocity(proc_gaze)
    saccades  = et.find_sac_from_fix(proc_gaze, fixations)
    et.char_saccades(saccades, px2deg)

    return fixations, saccades, proc_gaze.reset_index(drop=True)

In [4]:
# ── Main loop ──────────────────────────────────────────────────────────────────
def main():
    log.info("Processing %d participant(s).", len(PARTICIPANTS))

    for pid in PARTICIPANTS:
        print(f"Starting {pid}...")
        log.info("── %s ──────────────────────────────────────────", pid)

        eye_dir   = os.path.join(BASE_DIR, pid, f"{pid}_session1", f"eye_{pid}")
        gaze_path = os.path.join(eye_dir, "gaze_marked.csv")

        if not os.path.exists(gaze_path):
            log.error("  gaze_marked.csv not found: %s", gaze_path)
            continue

        try:
            # 1. Load
            gaze = load_and_clean(gaze_path)
            log.info("  Loaded %d samples", len(gaze))

            # 2. Validity check
            validity = compute_validity_rate(gaze)
            log.info("  Validity: %.1f%%", validity * 100)
            if validity < 0.55:
                log.warning("  EXCLUDED — validity below 55%% threshold")
                continue

            # 3. Run pipeline
            fixations, saccades, proc_gaze = run_pipeline(gaze)
            log.info(
                "  Fixations: %d   Saccades: %d",
                len(fixations), int((saccades.sclass == "S").sum())
            )

            # 4. Save
            if SAVE_FIXATIONS:
                p = os.path.join(eye_dir, "fixations.csv")
                fixations.to_csv(p, index=False)
                log.info("  Saved fixations  → %s", p)

            if SAVE_SACCADES:
                p = os.path.join(eye_dir, "saccades.csv")
                saccades.to_csv(p, index=False)
                log.info("  Saved saccades   → %s", p)

        except Exception:
            log.error("  FAILED:\n%s", traceback.format_exc())

    log.info("All done.")


main()


10:09:09  INFO      Processing 1 participant(s).
10:09:09  INFO      ── P12 ──────────────────────────────────────────


Starting P12...


10:09:43  INFO        Loaded 850215 samples
10:09:43  INFO        Validity: 86.5%


I2MC: Searching for valid interpolation windows
I2MC: Replace interpolation windows with Steffen interpolation
I2MC: 2-Means clustering started for left eye signal
I2MC: 2-Means clustering started for right eye signal
I2MC: Determining fixations based on clustering weight mean for averaged signal and separate eyes + 2.00*std


/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/di

READ: There are 18805 saccades, made up of 9744 forward, 7430 regressions and 1631 newlines


10:44:33  INFO        Saved fixations  → /content/drive/MyDrive/CAMES/data_collection_training/P12/P12_session1/eye_P12/fixations.csv
10:44:33  INFO        Saved saccades   → /content/drive/MyDrive/CAMES/data_collection_training/P12/P12_session1/eye_P12/saccades.csv
10:44:33  INFO      All done.
